In [0]:
import dlt
from pyspark.sql.functions import (
    col, split, substring, length, regexp_replace, when, expr, cast, current_date, datediff, trim, regexp_extract,
    concat_ws, lit, current_timestamp, row_number, desc, coalesce, count, sum, avg, max, min, stddev,
    countDistinct, date_format, date_sub, hour, dayofweek, month, year
)
from pyspark.sql.types import DecimalType, IntegerType, DateType
from pyspark.sql.window import Window

In [0]:
env = spark.conf.get("pipeline.env") 
catalog = "raiffka_catalog"
# Bronze schema is always "bronze" - no environment prefix
bronze_schema = "bronze"
# Silver schema uses environment prefix
silver_schema = f"{env}_silver" if env else "silver"
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE SCHEMA {silver_schema}")

In [0]:
#--------- CUSTOMERS ----------#
customers_expectations = {
    "valid_customer_id":"customer_id is not null",
    "valid_datum_narozeni":"year(datum_narozeni) > year(current_date())-100"
}

customer_expectations_expr = "NOT({0})".format(" AND ".join(customers_expectations.values()))

@dlt.table(
    name = "customers_clean_silver",
    comment = "This table contains the cleaned customers data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(customers_expectations)
def customers_clean_silver():
    df_customer = dlt.readStream(f"{catalog}.{bronze_schema}.customers_bronze")
    df_base = (
        df_customer
        .withColumn("cela_adresa", col("adresa"))
        .withColumn("zip_code", split(col("adresa"), ",").getItem(2))
        .withColumn("mesto", split(col("adresa"), ",").getItem(1))
        .withColumn("ulice", split(col("adresa"), ",").getItem(0))
        .withColumn("datum_narozeni", col("datum_narozeni").cast(DateType()))
        .withColumn(
            "predcisli",
            when(
                length(regexp_replace(col("tel_cislo"), " ", "")) > 9,
                split(col("tel_cislo"), " ").getItem(0)
            ).otherwise(None)
        )
        .withColumn(
            "tel_cislo",
            substring(regexp_replace(col("tel_cislo"), " ", ""), -9, 9)
        )
        .withColumn("pocet_clenu_domacnost", col("pocet_clenu_domacnost").cast(IntegerType()))
        .withColumn("datum_prvniho_otevreni_uctu", col("datum_prvniho_otevreni_uctu").cast(DateType()))
        .withColumn("prijem", col("prijem").cast(DecimalType(20,2)))
        .withColumn("sum_prm_uspor_12m", col("sum_prm_uspor_12m").cast(DecimalType(20,2)))
        .withColumn("min_atm_amt", col("min_atm_amt").cast(DecimalType(20,2)))
        .withColumn("pocet_nemovitosti", col("pocet_nemovitosti").cast(IntegerType()))
    )
        
    return (
        df_base
        .withColumn("ulice", split(col("ulice"), " ").getItem(0))
        .withColumn("cislo_popisne", split(col("ulice"), " ").getItem(1))
        .withColumn("client_age", expr("year(current_date()) - year(datum_narozeni)"))
        .withColumn("days_from_open_first_acc", datediff(current_date(), col("datum_prvniho_otevreni_uctu")))
        .withColumn("is_quarantined",expr(customer_expectations_expr))
        .select(
            "customer_id",
            "jmeno",
            "prijmeni",
            "cela_adresa",
            "zip_code",
            "mesto",
            "ulice",
            "cislo_popisne",
            "email",
            "rodne_cislo",
            "stav",
            "predcisli",
            "tel_cislo",
            "datum_narozeni",
            "typ_bydleni",
            "pracovni_stav",
            "pohlavi",
            "pocet_clenu_domacnost",
            "datum_prvniho_otevreni_uctu",
            "zeme",
            "prijem",
            "titul_pred",
            "titul_za",
            "sum_prm_uspor_12m",
            "min_atm_amt",
            "rezident",
            "pocet_nemovitosti",
            "client_age",
            "days_from_open_first_acc",
            "snapshot_date",
            "is_quarantined"
        )
    )

@dlt.table(name='customers_clean_good_records',comment='customers cleanded and validate data')
def customer_clean():
    df_customer = dlt.readStream('customers_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='customers_clean_bad_records',comment='customers cleaned and bad data')
def customer_clean():
    df_customer = dlt.readStream('customers_clean_silver')
    return (
        df_customer
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_customers",
        comment="cusotmers dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_customers",
        source = "customers_clean_good_records",
        keys = ["customer_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )


#------------------------------- ACCOUNT BALANCE -------------------------
@dlt.table(
    name = "account_balances_silver",
    comment = "This table contains the cleaned account balances data",
    table_properties={"quality": "silver"}
)
def account_balances_silver():
    df_ab = dlt.readStream(f"{catalog}.{bronze_schema}.account_balances_bronze")
    return (
        df_ab
        .withColumn("balance", col("balance").cast(DecimalType(20,2)))
        .withColumn("datum", col("datum").cast(DateType()))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- ACCOUNTS -------------------------
accounts_expectations = {
    "valid_account":"typ_uctu is not null and stav is not null",
    "valid_mena":"mena is not null",
}

accounts_expectations_expr = "NOT({0})".format(" AND ".join(accounts_expectations.values()))

@dlt.table(
    name = "accounts_clean_silver",
    comment = "This table contains the cleaned accounts data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(accounts_expectations)
def accounts_clean_silver():
    df_accounts = dlt.readStream(f"{catalog}.{bronze_schema}.accounts_bronze")
    return (
        df_accounts
        .withColumn("is_quarantined",expr(accounts_expectations_expr))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

@dlt.table(name='accounts_clean_good_records',comment='accounts cleanded and validate data')
def accounts_clean_good_records():
    df_accounts = dlt.readStream('accounts_clean_silver')
    return (
        df_accounts
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='accounts_clean_bad_records',comment='accounts cleaned and bad data')
def accounts_clean_bad_records():
    df_accounts = dlt.readStream('accounts_clean_silver')
    return (
        df_accounts
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_accounts",
        comment="accounts dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_accounts",
        source = "accounts_clean_good_records",
        keys = ["account_id","customer_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )

#------------------------------- CARD TRANSACTIONS -------------------------
@dlt.table(
    name="card_transactions_silver",
    comment="This table contains the cleaned card transactions data",
    table_properties={"quality": "silver"}
)
def card_transactions_silver():
    df_ct = dlt.readStream(f"{catalog}.{bronze_schema}.card_transactions_bronze")
    return(
        df_ct
        .withColumn("amount", col("amount").cast(DecimalType(20,2)))
        .withColumn("datum", col("datum").cast(DateType()))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- CARDS  -------------------------
cards_expectations = {
    "valid_card":"card_id is not null and account_id is not null and typ_karty is not null",
}

cards_expectations_expr = "NOT({0})".format(" AND ".join(cards_expectations.values()))

@dlt.table(
    name = "cards_clean_silver",
    comment = "This table contains the cleaned cards data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(cards_expectations)
def cards_clean_silver():
    df_cards = dlt.readStream(f"{catalog}.{bronze_schema}.cards_bronze")
    return (
        df_cards
        .withColumn("platnost_do",col("platnost_do").cast(DateType()))
        .withColumn("is_quarantined",expr(cards_expectations_expr))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

@dlt.table(name='cards_clean_good_records',comment='cards cleanded and validate data')
def cards_clean_good_records():
    df_cards = dlt.readStream('cards_clean_silver')
    return (
        df_cards
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='cards_clean_bad_records',comment='cards cleaned and bad data')
def cards_clean_bad_records():
    df_cards = dlt.readStream('cards_clean_silver')
    return (
        df_cards
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_cards",
        comment="cards dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_cards",
        source = "cards_clean_good_records",
        keys = ["card_id","account_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )


#------------------------------- EMPLOYEES  -------------------------
employees_expectations = {
    "valid_employee":"employee_id is not null and prijmeni is not null and role is not null",
}

employees_expectations_expr = "NOT({0})".format(" AND ".join(employees_expectations.values()))

@dlt.table(
    name = "employees_clean_silver",
    comment = "This table contains the cleaned employees data",
    partition_cols =["is_quarantined"]
)
@dlt.expect_all(employees_expectations)
def employees_clean_silver():
    df_employees = dlt.readStream(f"{catalog}.{bronze_schema}.employees_bronze")
    return (
        df_employees
        .withColumn("is_quarantined",expr(employees_expectations_expr))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

@dlt.table(name='employees_clean_good_records',comment='employees cleanded and validate data')
def employees_clean_good_records():
    df_employees = dlt.readStream('employees_clean_silver')
    return (
        df_employees
        .filter("is_quarantined=false")
        .drop("is_quarantined")
    )

@dlt.table(name='employees_clean_bad_records',comment='employees cleaned and bad data')
def employees_clean_bad_records():
    df_employees = dlt.readStream('employees_clean_silver')
    return (
        df_employees
        .filter("is_quarantined=true")
        .drop("is_quarantined")
    )

dlt.create_streaming_table(
        name="dim_employees",
        comment="employees dim table with SCD2 approach",
        table_properties={"quality": "silver"}
        )
dlt.apply_changes(
        target = "dim_employees",
        source = "employees_clean_good_records",
        keys = ["employee_id"],
        sequence_by = col("snapshot_date"),
        ignore_null_updates=False,
        stored_as_scd_type="2",
        track_history_except_column_list=['snapshot_date']
    )

#------------------------------- FINANCIAL TRANSACTIONS  -------------------------
@dlt.table(
    name="financial_transactions_silver",
    comment="This table contains the cleaned financial transactions data",
    table_properties={"quality": "silver"}
)
def financial_transactions_silver():
    df_ft = dlt.readStream(f"{catalog}.{bronze_schema}.financial_transactions_bronze")
    return(
        df_ft
        .withColumn("amount", col("amount").cast(DecimalType(20,2)))
        .withColumn("datum", col("datum").cast(DateType()))
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- MERCHATS  -------------------------
@dlt.table(
    name="merchants_silver",
    comment="This table contains the cleaned merchants data",
    table_properties={"quality": "silver"}
)
def merchants_silver():
    df_m = dlt.read(f"{catalog}.{bronze_schema}.merchants_bronze")
    return(
        df_m
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- PRODUCTS  -------------------------
@dlt.table(
    name="products_silver",
    comment="This table contains the cleaned products data",
    table_properties={"quality": "silver"}
)
def products_silver():
    df_p = dlt.read(f"{catalog}.{bronze_schema}.products_bronze")
    return(
        df_p
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- REGIONS  -------------------------
@dlt.table(
    name="regions_silver",
    comment="This table contains the cleaned regions data",
    table_properties={"quality": "silver"}
)
def regions_silver():
    df_p = dlt.read(f"{catalog}.{bronze_schema}.regions_bronze")
    return(
        df_p
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- CITIES  -------------------------
@dlt.table(
    name="cities_silver",
    comment="This table contains the cleaned cities data",
    table_properties={"quality": "silver"}
)
def cities_silver():
    df_c = dlt.read(f"{catalog}.{bronze_schema}.cities_bronze")
    return(
        df_c
        .drop("_rescued_data","source_file","ingestion_ts")
    )

#------------------------------- BRANCHES  -------------------------
@dlt.table(
    name="branches_silver",
    comment="This table contains the cleaned branches data",
    table_properties={"quality": "silver"}
)
def branches_silver():
    cp_reg = r'(\d+[A-Za-z]?(\s*/\s*\d+[A-Za-z]?)?)\s*$'
    df_b = dlt.read(f"{catalog}.{bronze_schema}.branches_bronze")
    return (
        df_b
        .withColumn("zip_code", split(col("adresa"), ",").getItem(2))
        .withColumn("mesto", split(col("adresa"), ",").getItem(1))
        .withColumn("_ulice", split(col("adresa"), ",").getItem(0))
        .withColumn("cislo_popisne", trim(regexp_extract(col("_ulice"), cp_reg, 1)))
        .withColumn("ulice", trim(regexp_replace(col("_ulice"), cp_reg, "")))
        .drop("_ulice","_rescued_data","source_file","ingestion_ts")
)

#===============================================================================
#                      ENTERPRISE DIMENSIONAL MODELING
#===============================================================================

#------------------------------- GEOGRAPHY HIERARCHY -------------------------
@dlt.table(
    name="dim_geography_hierarchy",
    comment="Geography dimension with complete hierarchy - Region > City > Branch",
    table_properties={"quality": "silver"}
)
def dim_geography_hierarchy():
    regions = dlt.read("regions_silver")
    cities = dlt.read("cities_silver")
    branches = dlt.read("branches_silver")
    
    # Complete geography hierarchy with denormalized structure for performance
    return (
        branches.alias("b")
        .join(cities.alias("c"), col("b.city_id") == col("c.city_id"), "inner")
        .join(regions.alias("r"), col("c.region_id") == col("r.region_id"), "inner")
        .select(
            col("b.branch_id"),
            col("b.nazev").alias("branch_name"),
            col("b.adresa").alias("branch_address"),
            col("b.ulice").alias("branch_street"),
            col("b.cislo_popisne").alias("branch_street_number"),
            col("b.zip_code").alias("branch_zip_code"),
            col("b.mesto").alias("branch_city_name"),
            col("c.city_id"),
            col("c.nazev").alias("city_name"),
            col("r.region_id"),
            col("r.nazev").alias("region_name"),
            # Business keys for hierarchy navigation
            concat_ws("-", col("r.region_id"), col("c.city_id"), col("b.branch_id")).alias("geography_key"),
            # Hierarchy levels for drill-down/roll-up
            lit("Branch").alias("geography_level"),
            current_timestamp().alias("created_at")
        )
    )

#------------------------------- CUSTOMER PRODUCT BRIDGE -------------------------
@dlt.table(
    name="bridge_customer_accounts_products",
    comment="Bridge table linking customers to their accounts and products with business metrics",
    table_properties={"quality": "silver"}
)
def bridge_customer_accounts_products():
    accounts = dlt.readStream("dim_accounts")
    customers = dlt.readStream("dim_customers")
    products = dlt.read("products_silver")
    account_balances = dlt.readStream("account_balances_silver")
    
    # Get latest balance for each account
    latest_balances = (
        account_balances
        .withColumn("row_num", row_number().over(
            Window.partitionBy("account_id").orderBy(desc("datum"))
        ))
        .filter(col("row_num") == 1)
        .select("account_id", "balance", col("datum").alias("balance_date"))
    )
    
    return (
        customers.alias("c")
        .filter(col("c.__END_AT").isNull())  # Current records only
        .join(accounts.alias("a"), col("c.customer_id") == col("a.customer_id"), "inner")
        .filter(col("a.__END_AT").isNull())  # Current records only
        .join(products.alias("p"), 
              when(col("a.typ_uctu") == "běžný účet", "PRD004")
              .when(col("a.typ_uctu") == "spořicí účet", "PRD001") 
              .when(col("a.typ_uctu") == "hypotéka", "PRD002")
              .when(col("a.typ_uctu") == "spotřebitelský úvěr", "PRD003")
              .otherwise("PRD004") == col("p.product_id"), "left")
        .join(latest_balances.alias("b"), col("a.account_id") == col("b.account_id"), "left")
        .select(
            col("c.customer_id"),
            col("a.account_id"),
            col("p.product_id"),
            col("a.typ_uctu").alias("account_type"),
            col("a.mena").alias("currency"),
            col("a.stav").alias("account_status"),
            col("p.nazev").alias("product_name"),
            col("p.typ").alias("product_type"),
            coalesce(col("b.balance"), lit(0)).alias("current_balance"),
            col("b.balance_date"),
            # Business metrics
            when(col("a.stav") == "aktivní", 1).otherwise(0).alias("is_active_account"),
            when(col("b.balance") > 0, 1).otherwise(0).alias("has_positive_balance"),
            # Customer segmentation flags
            when(col("c.prijem") > 100000, "High Income")
            .when(col("c.prijem") > 50000, "Medium Income")
            .otherwise("Low Income").alias("income_segment"),
            # Account vintage
            datediff(current_date(), col("c.datum_prvniho_otevreni_uctu")).alias("days_since_first_account"),
            current_timestamp().alias("created_at")
        )
    )

#------------------------------- TRANSACTION ENRICHMENT -------------------------
@dlt.table(
    name="fact_transactions_enriched",
    comment="Enriched transaction fact table with customer, merchant, and geography context",
    table_properties={"quality": "silver"}
)
def fact_transactions_enriched():
    card_trans = dlt.readStream("card_transactions_silver")
    financial_trans = dlt.readStream("financial_transactions_silver")
    customers = dlt.readStream("dim_customers")
    accounts = dlt.readStream("dim_accounts")
    cards = dlt.readStream("dim_cards")
    merchants = dlt.read("merchants_silver")
    geography = dlt.read("dim_geography_hierarchy")
    
    # Union card and financial transactions with common schema
    card_transactions_unified = (
        card_trans.alias("card_trans")
        .join(cards.filter(col("__END_AT").isNull()).alias("card"), 
              col("card_trans.card_id") == col("card.card_id"), "inner")
        .join(accounts.filter(col("__END_AT").isNull()).alias("acc"), 
              col("card.account_id") == col("acc.account_id"), "inner")
        .join(customers.filter(col("__END_AT").isNull()).alias("cust"), 
              col("acc.customer_id") == col("cust.customer_id"), "inner")
        .join(merchants.alias("merch"), 
              col("card_trans.merchant_id") == col("merch.merchant_id"), "left")
        .join(geography.alias("geo"), 
              col("merch.city_id") == col("geo.city_id"), "left")
        .select(
            col("card_trans.card_transaction_id").alias("transaction_id"),
            lit("CARD").alias("transaction_type"),
            col("card_trans.card_id"),
            col("acc.account_id"),
            col("cust.customer_id"),
            col("card_trans.merchant_id"),
            col("card_trans.amount"),
            col("card_trans.datum").alias("transaction_date"),
            col("merch.kategorie").alias("merchant_category"),
            col("geo.region_name"),
            col("geo.city_name"),
            # Business metrics
            when(col("card_trans.amount") > 1000, "High Value")
            .when(col("card_trans.amount") > 100, "Medium Value")
            .otherwise("Low Value").alias("transaction_value_segment"),
            hour(col("card_trans.datum")).alias("transaction_hour"),
            dayofweek(col("card_trans.datum")).alias("transaction_day_of_week"),
            month(col("card_trans.datum")).alias("transaction_month"),
            year(col("card_trans.datum")).alias("transaction_year")
        )
    )
    
    financial_transactions_unified = (
        financial_trans.alias("financial_trans")
        .join(accounts.filter(col("__END_AT").isNull()).alias("acc"), 
              col("financial_trans.account_id") == col("acc.account_id"), "inner")
        .join(customers.filter(col("__END_AT").isNull()).alias("cust"), 
              col("acc.customer_id") == col("cust.customer_id"), "inner")
        .select(
            col("financial_trans.transaction_id"),
            lit("FINANCIAL").alias("transaction_type"),
            lit(None).cast("string").alias("card_id"),
            col("acc.account_id"),
            col("cust.customer_id"),
            lit(None).cast("string").alias("merchant_id"),
            col("financial_trans.amount"),
            col("financial_trans.datum").alias("transaction_date"),
            col("financial_trans.typ").alias("merchant_category"),
            lit(None).cast("string").alias("region_name"),
            lit(None).cast("string").alias("city_name"),
            # Business metrics
            when(col("financial_trans.amount") > 1000, "High Value")
            .when(col("financial_trans.amount") > 100, "Medium Value")
            .otherwise("Low Value").alias("transaction_value_segment"),
            hour(col("financial_trans.datum")).alias("transaction_hour"),
            dayofweek(col("financial_trans.datum")).alias("transaction_day_of_week"),
            month(col("financial_trans.datum")).alias("transaction_month"),
            year(col("financial_trans.datum")).alias("transaction_year")
        )
    )
    
    return card_transactions_unified.union(financial_transactions_unified)

#------------------------------- CUSTOMER RISK SCORING -------------------------
@dlt.table(
    name="customer_risk_metrics_silver",
    comment="Customer risk scoring and behavioral analytics for enterprise decision making",
    table_properties={"quality": "silver"}
)
def customer_risk_metrics_silver():
    customers = dlt.readStream("dim_customers")
    accounts = dlt.readStream("dim_accounts")
    # Use simple transaction tables instead of enriched fact to avoid circular dependency
    card_trans = dlt.readStream("card_transactions_silver")
    financial_trans = dlt.readStream("financial_transactions_silver")
    balances = dlt.readStream("account_balances_silver")
    
    # Combine transactions for risk analysis
    all_transactions = (
        card_trans.select("card_transaction_id", "amount", "datum", lit("CARD").alias("type"))
        .withColumnRenamed("card_transaction_id", "transaction_id")
        .join(dlt.readStream("dim_cards").filter(col("__END_AT").isNull()), "card_id", "inner")
        .join(dlt.readStream("dim_accounts").filter(col("__END_AT").isNull()), "account_id", "inner")
        .select("transaction_id", "customer_id", "amount", "datum", "type")
        .union(
            financial_trans.select("transaction_id", "amount", "datum", lit("FINANCIAL").alias("type"))
            .join(dlt.readStream("dim_accounts").filter(col("__END_AT").isNull()), "account_id", "inner")
            .select("transaction_id", "customer_id", "amount", "datum", "type")
        )
    )
    
    # Calculate transaction metrics per customer
    customer_transaction_metrics = (
        all_transactions
        .filter(col("datum") >= date_sub(current_date(), 90))  # Last 90 days
        .groupBy("customer_id")
        .agg(
            count("*").alias("transaction_count_90d"),
            sum("amount").alias("total_transaction_amount_90d"),
            avg("amount").alias("avg_transaction_amount_90d"),
            max("amount").alias("max_transaction_amount_90d"),
            countDistinct("type").alias("unique_categories_90d"),
            countDistinct(date_format("datum", "yyyy-MM-dd")).alias("active_days_90d")
        )
    )
    
    # Calculate balance volatility
    balance_metrics = (
        balances
        .filter(col("datum") >= date_sub(current_date(), 180))  # Last 6 months
        .groupBy("account_id")
        .agg(
            avg("balance").alias("avg_balance_6m"),
            stddev("balance").alias("balance_volatility_6m"),
            min("balance").alias("min_balance_6m"),
            max("balance").alias("max_balance_6m")
        )
    )
    
    # Aggregate balance metrics per customer
    customer_balance_metrics = (
        balance_metrics.alias("bm")
        .join(accounts.filter(col("__END_AT").isNull()).alias("acc"), 
              col("bm.account_id") == col("acc.account_id"), "inner")
        .groupBy("acc.customer_id")
        .agg(
            sum("avg_balance_6m").alias("total_avg_balance_6m"),
            avg("balance_volatility_6m").alias("avg_balance_volatility_6m"),
            min("min_balance_6m").alias("overall_min_balance_6m"),
            max("max_balance_6m").alias("overall_max_balance_6m"),
            count("*").alias("active_accounts_count")
        )
    )
    
    return (
        customers.alias("c")
        .filter(col("c.__END_AT").isNull())  # Current records only
        .join(customer_transaction_metrics.alias("tm"), 
              col("c.customer_id") == col("tm.customer_id"), "left")
        .join(customer_balance_metrics.alias("bm"), 
              col("c.customer_id") == col("bm.customer_id"), "left")
        .select(
            col("c.customer_id"),
            col("c.jmeno"),
            col("c.prijmeni"),
            col("c.client_age"),
            col("c.prijem"),
            col("c.days_from_open_first_acc"),
            # Transaction behavior metrics
            coalesce(col("tm.transaction_count_90d"), lit(0)).alias("transaction_count_90d"),
            coalesce(col("tm.total_transaction_amount_90d"), lit(0)).alias("total_transaction_amount_90d"),
            coalesce(col("tm.avg_transaction_amount_90d"), lit(0)).alias("avg_transaction_amount_90d"),
            coalesce(col("tm.unique_categories_90d"), lit(0)).alias("unique_categories_90d"),
            coalesce(col("tm.active_days_90d"), lit(0)).alias("active_days_90d"),
            # Balance behavior metrics
            coalesce(col("bm.total_avg_balance_6m"), lit(0)).alias("total_avg_balance_6m"),
            coalesce(col("bm.avg_balance_volatility_6m"), lit(0)).alias("avg_balance_volatility_6m"),
            coalesce(col("bm.active_accounts_count"), lit(0)).alias("active_accounts_count"),
            # Risk scoring components
            when(col("c.prijem") < 30000, 3)
            .when(col("c.prijem") < 60000, 2)
            .otherwise(1).alias("income_risk_score"),
            
            when(coalesce(col("bm.avg_balance_volatility_6m"), lit(0)) > 50000, 3)
            .when(coalesce(col("bm.avg_balance_volatility_6m"), lit(0)) > 20000, 2)
            .otherwise(1).alias("volatility_risk_score"),
            
            when(coalesce(col("tm.transaction_count_90d"), lit(0)) < 5, 3)
            .when(coalesce(col("tm.transaction_count_90d"), lit(0)) < 20, 2)
            .otherwise(1).alias("activity_risk_score"),
            
            current_timestamp().alias("calculated_at")
        )
        .withColumn("composite_risk_score", 
                   col("income_risk_score") + col("volatility_risk_score") + col("activity_risk_score"))
        .withColumn("risk_category",
                   when(col("composite_risk_score") >= 8, "High Risk")
                   .when(col("composite_risk_score") >= 6, "Medium Risk")
                   .otherwise("Low Risk"))
    )